In [0]:
 # Databricks notebook source

# ============================================================
# 03_GOLD
# Pipeline de Dados - Inside Airbnb São Paulo
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
 # COMMAND ----------

# ============================================================
# LEITURA DA SILVER
# ============================================================

SILVER_TABLE = "workspace.default.silver_listings"
 
df_silver = spark.table(SILVER_TABLE)

print("Registros Silver:", df_silver.count())
print("Atributos Silver:", len(df_silver.columns))

In [0]:
 # COMMAND ----------

# ============================================================
# BASE GOLD
# ============================================================

df_gold_base = df_silver.select(
    "id",
    "neighbourhood_cleansed",
    "room_type",
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating",
    "host_is_superhost",
    "instant_bookable",
    "last_scraped"
)

print("Registros base Gold:", df_gold_base.count())

display(df_gold_base.limit(10))

In [0]:
 # COMMAND ----------

# ============================================================
# FACT_LISTING
# ============================================================

fact_listing = df_gold_base

FACT_TABLE = "workspace.default.fact_listing"

(
    fact_listing.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(FACT_TABLE)
)

print(f"Tabela criada: {FACT_TABLE}")

In [0]:
 # COMMAND ----------

# ============================================================
# VALIDAÇÃO FACT_LISTING
# ============================================================

df_fact = spark.table(FACT_TABLE)

print("Registros:", df_fact.count())
print("Atributos:", len(df_fact.columns))

display(df_fact.limit(10))

In [0]:
  
  # COMMAND ----------

# ============================================================
# DIM_NEIGHBOURHOOD
# ============================================================

DIM_NEIGHBOURHOOD = "workspace.default.dim_neighbourhood"

dim_neighbourhood = (
    df_fact
    .select("neighbourhood_cleansed")
    .distinct()
    .where(F.col("neighbourhood_cleansed").isNotNull())
    .withColumn(
        "neighbourhood_key",
        F.row_number().over(
            Window.orderBy("neighbourhood_cleansed")
        )
    )
    .select(
        "neighbourhood_key",
        "neighbourhood_cleansed"
    )
)

(
    dim_neighbourhood.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(DIM_NEIGHBOURHOOD)
)

print(
    f"Tabela criada: {DIM_NEIGHBOURHOOD}"
)

print(
    "Bairros:",
    dim_neighbourhood.count()
)

display(dim_neighbourhood)


In [0]:
 # COMMAND ----------

# ============================================================
# DIM_ROOM_TYPE
# ============================================================

DIM_ROOM_TYPE = "workspace.default.dim_room_type"

dim_room_type = (
    df_fact
    .select("room_type")
    .distinct()
    .where(F.col("room_type").isNotNull())
    .withColumn(
        "room_type_key",
        F.row_number().over(
            Window.orderBy("room_type")
        )
    )
    .select(
        "room_type_key",
        "room_type"
    )
)

(
    dim_room_type.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(DIM_ROOM_TYPE)
)

print(f"Tabela criada: {DIM_ROOM_TYPE}")
print("Tipos:", dim_room_type.count())

display(dim_room_type)

In [0]:
 
 # COMMAND ----------

# ============================================================
# FACT_LISTING_DIMENSIONAL
# ============================================================

fact_listing_dimensional = (
    df_fact.alias("f")
    .join(
        dim_neighbourhood.alias("n"),
        F.col("f.neighbourhood_cleansed")
        == F.col("n.neighbourhood_cleansed"),
        "left"
    )
    .join(
        dim_room_type.alias("r"),
        F.col("f.room_type")
        == F.col("r.room_type"),
        "left"
    )
    .select(
        F.col("f.id").alias("listing_id"),
        F.col("n.neighbourhood_key"),
        F.col("f.neighbourhood_cleansed"),
        F.col("r.room_type_key"),
        F.col("f.room_type"),
        F.col("f.price"),
        F.col("f.accommodates"),
        F.col("f.bedrooms"),
        F.col("f.beds"),
        F.col("f.availability_30"),
        F.col("f.availability_60"),
        F.col("f.availability_90"),
        F.col("f.availability_365"),
        F.col("f.number_of_reviews"),
        F.col("f.review_scores_rating"),
        F.col("f.host_is_superhost"),
        F.col("f.instant_bookable"),
        F.col("f.last_scraped")
    )
)

DIM_FACT_TABLE = "workspace.default.fact_listing_dimensional"

(
    fact_listing_dimensional.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(DIM_FACT_TABLE)
)

print(f"Tabela criada: {DIM_FACT_TABLE}")
print("Registros:", fact_listing_dimensional.count())

display(fact_listing_dimensional.limit(10))

In [0]:
 # COMMAND ----------

# ============================================================
# GOLD 1 — PREÇO POR BAIRRO
# Pergunta 1
# ============================================================

gold_neighbourhood_price = (
    fact_listing_dimensional
    .where(
        (F.col("price") > 0) &
        F.col("neighbourhood_cleansed").isNotNull()
    )
    .groupBy("neighbourhood_cleansed")
    .agg(
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.count("*").alias("quantidade_anuncios")
    )
    .orderBy(F.desc("preco_medio"))
)

GOLD_NEIGHBOURHOOD = "workspace.default.gold_neighbourhood_price"

(
    gold_neighbourhood_price.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(GOLD_NEIGHBOURHOOD)
)

print(f"Tabela criada: {GOLD_NEIGHBOURHOOD}")
print("Bairros:", gold_neighbourhood_price.count())

display(gold_neighbourhood_price.limit(20))

In [0]:
 # COMMAND ----------

# ============================================================
# GOLD 2 — PREÇO POR TIPO DE ACOMODAÇÃO
# Pergunta 2
# ============================================================

gold_room_type_price = (
    fact_listing_dimensional
    .where(F.col("price") > 0)
    .groupBy("room_type")
    .agg(
        F.count("*").alias("quantidade_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio")
    )
    .orderBy(F.desc("preco_medio"))
)

GOLD_ROOM_TYPE = "workspace.default.gold_room_type_price"

(
    gold_room_type_price.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(GOLD_ROOM_TYPE)
)

display(gold_room_type_price)

In [0]:
 # COMMAND ----------

# ============================================================
# GOLD 3 — PREÇO POR CAPACIDADE
# Pergunta 3
# ============================================================

gold_preco_capacidade = (
    fact_listing_dimensional
    .where(
        (F.col("price") > 0) &
        F.col("accommodates").isNotNull()
    )
    .groupBy("accommodates")
    .agg(
        F.count("*").alias("quantidade_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.avg("bedrooms"), 2).alias("quartos_medios"),
        F.round(F.avg("beds"), 2).alias("camas_medias")
    )
    .orderBy("accommodates")
)

GOLD_CAPACITY = "workspace.default.gold_preco_capacidade"

(
    gold_preco_capacidade.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(GOLD_CAPACITY)
)

display(gold_preco_capacidade)

In [0]:
 # COMMAND ----------

# ============================================================
# GOLD 4 — AVALIAÇÃO × PREÇO
# Pergunta 4
# ============================================================

gold_review_analysis = (
    fact_listing_dimensional
    .where(
        (F.col("price") > 0) &
        F.col("review_scores_rating").isNotNull()
    )
    .select(
        "price",
        "review_scores_rating",
        "number_of_reviews"
    )
)

GOLD_REVIEW = "workspace.default.gold_review_analysis"

(
    gold_review_analysis.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(GOLD_REVIEW)
)

display(gold_review_analysis.limit(20))

In [0]:
 # COMMAND ----------

# ============================================================
# CORRELAÇÕES
# Perguntas 3, 4 e 5
# ============================================================

df_corr = fact_listing_dimensional

corr_accommodates = (
    df_corr
    .where(
        F.col("price").isNotNull() &
        F.col("accommodates").isNotNull() &
        (F.col("price") > 0)
    )
    .stat.corr("accommodates", "price")
)

corr_bedrooms = (
    df_corr
    .where(
        F.col("price").isNotNull() &
        F.col("bedrooms").isNotNull() &
        (F.col("price") > 0)
    )
    .stat.corr("bedrooms", "price")
)

corr_beds = (
    df_corr
    .where(
        F.col("price").isNotNull() &
        F.col("beds").isNotNull() &
        (F.col("price") > 0)
    )
    .stat.corr("beds", "price")
)

corr_rating = (
    df_corr
    .where(
        F.col("price").isNotNull() &
        F.col("review_scores_rating").isNotNull() &
        (F.col("price") > 0)
    )
    .stat.corr("review_scores_rating", "price")
)

corr_availability = (
    df_corr
    .where(
        F.col("price").isNotNull() &
        F.col("availability_365").isNotNull() &
        (F.col("price") > 0)
    )
    .stat.corr("availability_365", "price")
)

print(f"accommodates × price: {corr_accommodates:.4f}")
print(f"bedrooms × price:     {corr_bedrooms:.4f}")
print(f"beds × price:         {corr_beds:.4f}")
print(f"rating × price:       {corr_rating:.4f}")
print(f"availability × price: {corr_availability:.4f}")

In [0]:
 # COMMAND ----------

# ============================================================
# GOLD 5 — DISPONIBILIDADE
# Pergunta 5
# ============================================================

gold_availability = (
    fact_listing_dimensional
    .select(
        "neighbourhood_cleansed",
        "room_type",
        F.col("availability_30").cast("double").alias("availability_30"),
        F.col("availability_60").cast("double").alias("availability_60"),
        F.col("availability_90").cast("double").alias("availability_90"),
        "availability_365"
    )
    .where(
        F.col("neighbourhood_cleansed").isNotNull() &
        F.col("room_type").isNotNull()
    )
    .groupBy(
        "neighbourhood_cleansed",
        "room_type"
    )
    .agg(
        F.count("*").alias("quantidade_anuncios"),
        F.round(F.avg("availability_30"), 2).alias("media_disponibilidade_30"),
        F.round(F.avg("availability_60"), 2).alias("media_disponibilidade_60"),
        F.round(F.avg("availability_90"), 2).alias("media_disponibilidade_90"),
        F.round(F.avg("availability_365"), 2).alias("media_disponibilidade_365")
    )
)

GOLD_AVAILABILITY = "workspace.default.gold_availability"

(
    gold_availability.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(GOLD_AVAILABILITY)
)

print(f"Tabela criada: {GOLD_AVAILABILITY}")

display(
    gold_availability
    .where(F.col("quantidade_anuncios") >= 20)
    .orderBy(F.desc("media_disponibilidade_30"))
    .limit(20)
)

In [0]:
 # COMMAND ----------

# ============================================================
# GOLD 6 — ÍNDICE DE OPORTUNIDADE
# Pergunta 6
# ============================================================

df_opportunity_base = (
    fact_listing_dimensional
    .select(
        "listing_id",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "review_scores_rating",
        "availability_365"
    )
    .where(
        (F.col("price") > 0) &
        F.col("review_scores_rating").isNotNull() &
        F.col("availability_365").isNotNull()
    )
)

stats = df_opportunity_base.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.min("review_scores_rating").alias("min_rating"),
    F.max("review_scores_rating").alias("max_rating"),
    F.min("availability_365").alias("min_availability"),
    F.max("availability_365").alias("max_availability")
).collect()[0]

min_price = stats["min_price"]
max_price = stats["max_price"]

min_rating = stats["min_rating"]
max_rating = stats["max_rating"]

min_availability = stats["min_availability"]
max_availability = stats["max_availability"]

df_opportunity = (
    df_opportunity_base

    # Quanto maior a avaliação, melhor
    .withColumn(
        "score_rating",
        (F.col("review_scores_rating") - F.lit(min_rating))
        / F.lit(max_rating - min_rating)
    )

    # Quanto maior a disponibilidade, melhor
    .withColumn(
        "score_availability",
        (F.col("availability_365") - F.lit(min_availability))
        / F.lit(max_availability - min_availability)
    )

    # Quanto menor o preço, melhor
    .withColumn(
        "score_price",
        1 -
        (
            (F.col("price") - F.lit(min_price))
            / F.lit(max_price - min_price)
        )
    )

    .withColumn(
        "opportunity_index",
        (
            F.col("score_rating") * 0.40
            + F.col("score_availability") * 0.30
            + F.col("score_price") * 0.30
        )
    )
)

GOLD_OPPORTUNITY = "workspace.default.gold_opportunity"

(
    df_opportunity.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(GOLD_OPPORTUNITY)
)

display(
    df_opportunity
    .orderBy(F.desc("opportunity_index"))
    .limit(20)
)

In [0]:
 # COMMAND ----------

# ============================================================
# GOLD 7 — PERFIL DOS ANÚNCIOS
# ============================================================

gold_perfil_anuncio = (
    fact_listing_dimensional
    .groupBy(
        "room_type"
    )
    .agg(
        F.count("*").alias("quantidade_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.avg("review_scores_rating"), 2).alias("avaliacao_media"),
        F.round(F.avg("availability_365"), 2).alias(
            "disponibilidade_media_365"
        ),
        F.round(F.avg("accommodates"), 2).alias(
            "capacidade_media"
        ),
        F.round(F.avg("bedrooms"), 2).alias(
            "quartos_medios"
        ),
        F.round(F.avg("beds"), 2).alias(
            "camas_medias"
        )
    )
)

GOLD_PROFILE = "workspace.default.gold_perfil_anuncio"

(
    gold_perfil_anuncio.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(GOLD_PROFILE)
)

display(gold_perfil_anuncio)

In [0]:
 # COMMAND ----------

# ============================================================
# REPRESENTATIVIDADE
# ============================================================

gold_neighbourhood_representative = (
    gold_neighbourhood_price
    .where(F.col("quantidade_anuncios") >= 20)
    .orderBy(F.desc("preco_medio"))
)

print(
    "Bairros com pelo menos 20 anúncios:",
    gold_neighbourhood_representative.count()
)

display(
    gold_neighbourhood_representative.limit(20)
)

In [0]:
 # COMMAND ----------

# ============================================================
# CONSOLIDAÇÃO DAS PERGUNTAS DE NEGÓCIO
# ============================================================

print("============================================================")
print("ANÁLISE FINAL — INSIDE AIRBNB SÃO PAULO")
print("============================================================")

print("\nPERGUNTA 1")
print("Quais bairros apresentam os maiores e menores preços médios?")
display(
    gold_neighbourhood_representative.limit(10)
)

print("\nPERGUNTA 2")
print("Qual é a relação entre o tipo de acomodação e o preço?")
display(
    gold_room_type_price
)

print("\nPERGUNTA 3")
print("Qual é a relação entre capacidade, quartos, camas e preço?")
print(f"Correlação capacidade × preço: {corr_accommodates:.4f}")
print(f"Correlação quartos × preço:     {corr_bedrooms:.4f}")
print(f"Correlação camas × preço:       {corr_beds:.4f}")
display(
    gold_preco_capacidade
)

print("\nPERGUNTA 4")
print("Anúncios com melhores avaliações apresentam preços maiores?")
print(f"Correlação avaliação × preço: {corr_rating:.4f}")

print("\nPERGUNTA 5")
print("Como a disponibilidade varia por bairro e tipo de acomodação?")
display(
    gold_availability
    .where(F.col("quantidade_anuncios") >= 20)
    .orderBy(F.desc("media_disponibilidade_30"))
    .limit(15)
)

print("\nPERGUNTA 6")
print("Quais perfis combinam preço, avaliação e disponibilidade?")
display(
    df_opportunity
    .orderBy(F.desc("opportunity_index"))
    .limit(15)
)

In [0]:
 # COMMAND ----------

# ============================================================
# VALIDAÇÃO FINAL DO GOLD
# ============================================================

tabelas_gold = [
    "workspace.default.fact_listing",
    "workspace.default.fact_listing_dimensional",
    "workspace.default.dim_neighbourhood",
    "workspace.default.dim_room_type",
    "workspace.default.gold_neighbourhood_price",
    "workspace.default.gold_room_type_price",
    "workspace.default.gold_preco_capacidade",
    "workspace.default.gold_availability",
    "workspace.default.gold_review_analysis",
    "workspace.default.gold_opportunity",
    "workspace.default.gold_perfil_anuncio"
]

print("============================================================")
print("VALIDAÇÃO FINAL — CAMADA GOLD")
print("============================================================")

for tabela in tabelas_gold:
    df = spark.table(tabela)
    print(f"{tabela}: {df.count()} registros")

print("\nPIPELINE CONCLUÍDO.")